# 🚀 AIInfluencerOS — Kaggle Live GPU Worker (Diya Rai SDXL LoRA)
### Dual T4 GPU / P100 — 30 Hours Free / Week
1. **Settings (Right Panel):**
   - Accelerator: **GPU T4 x 2**
   - Internet: **ON**
2. **Add LoRA File:** Click `+ Add Input` -> Upload `diyarai_sdxl_lora.safetensors` as dataset, or upload directly to `/kaggle/working/`.

In [ ]:
# Cell 1: Install compatible diffusers and transformers
!pip install -q "transformers<4.48.0" "diffusers>=0.30.0" accelerate safetensors gradio
print('✅ Environment Ready!')

In [ ]:
# Cell 2: Launch Live Worker with Diya Rai LoRA
# Compatibility shim for transformers.utils
import transformers.utils as _tu
if not hasattr(_tu, 'FLAX_WEIGHTS_NAME'):
    _tu.FLAX_WEIGHTS_NAME = 'flax_model.msgpack'

import os, sys, glob, time, torch
from pathlib import Path
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import gradio as gr

# Locate LoRA
LORA_PATH = None
for p in ['/kaggle/input/diyarai-sdxl-lora/diyarai_sdxl_lora.safetensors', '/kaggle/working/diyarai_sdxl_lora.safetensors']:
    if os.path.exists(p):
        LORA_PATH = p
        break
if not LORA_PATH:
    f = glob.glob('/kaggle/input/**/diyarai*.safetensors', recursive=True)
    if f: LORA_PATH = f[0]

print(f'📦 Diya Rai LoRA: {LORA_PATH}')

# Load SDXL
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True
).to('cuda')
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)

if LORA_PATH:
    pipe.load_lora_weights(LORA_PATH)
    print('✅ LoRA Attached Successfully!')

def generate_diya(prompt, lora_scale=0.85, steps=28, guidance=7.0):
    if 'diyarai' not in prompt.lower():
        prompt = f'photorealistic 8k portrait of diyarai woman, {prompt}'
    image = pipe(
        prompt=prompt,
        negative_prompt='blurry, smooth plastic skin, 3d render, cartoon, deformed, low quality',
        cross_attention_kwargs={'scale': lora_scale},
        num_inference_steps=steps,
        guidance_scale=guidance,
        width=832,
        height=1216
    ).images[0]
    out_path = f'/kaggle/working/diya_{int(time.time())}.png'
    image.save(out_path)
    return image, out_path

demo = gr.Interface(
    fn=generate_diya,
    inputs=[gr.Textbox(label='Prompt', value='sitting in Bandra cafe wearing beige linen blazer, sipping iced latte, soft window sunlight, candid smile'), gr.Slider(0.1, 1.2, value=0.85, label='LoRA Scale')],
    outputs=[gr.Image(label='Output Portrait'), gr.File(label='Download PNG')],
    title='👩 Diya Rai GPU Engine'
)
demo.launch(share=True)